In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from datetime import datetime

In [2]:
#files = sorted(glob.glob('/home/ulyanov/data/solo/phi/polar/*'))
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/polar/*'))
print(len(files))
print(files[0], files[-1])

356
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_000000_TAI.3.magnetogram.fits /home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250530_220000_TAI.3.magnetogram.fits


In [13]:
nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 450#512

grid = np.mgrid[:nx,:ny].astype(np.float32)
view_new = View(nx, ny, xc, yc, Rsun, 0, 0, 0)

plt.ioff()

for i, file in enumerate(files[:]):
    print(file)

    with fits.open(file) as hdul:
        header = hdul[1].header.copy()
        data = hdul[1].data.copy()

    view = View.from_header(header)
    view_new = view_new.update(crln=view.crln, crlt=view.crlt)#, rsun=view.rsun)

    transform = view_new.to_spherical(correct_mu=True, mu_thr=0.1) - view.to_spherical(correct_mu=True, correct_dr=False, mu_thr=0.1)
    grid_, alpha = transform(grid)
    data = bilinear(data, *grid_) * alpha

    grid_, _ = (~view_new.to_spherical())(np.mgrid[-90:90:1,-180:180:1])
    grid_ = np.array(grid_)
    meridians = grid_[:,:,::15]
    parallels = np.transpose(grid_[:,::15,:], (0,2,1))

    fig, ax = plt.subplots(figsize=(10,10))
    im = ax.imshow(data, 'seismic', vmin=-100, vmax=100, origin='lower')

    ax.plot(meridians[1], meridians[0], color='black', ls='--', lw=0.5)#, alpha=0.5)
    ax.plot(parallels[1], parallels[0], color='black', ls='--', lw=0.5)#, alpha=0.5)

    cax = ax.inset_axes((0.8, 0.985, 0.2, 0.015))
    fig.colorbar(im, cax=cax, orientation='horizontal', label=r'$B_{los}$, G')
    ax.axis('off')

    ax.annotate(datetime.fromisoformat(header['DATE-OBS']).replace(microsecond=0), (0.7,0.05), size=16, xycoords='axes fraction')
    plt.tight_layout()

    #plt.savefig(f'temp/{file.split("/")[-1].split(".")[0]}.png')
    plt.savefig(f'temp/{file.split("/")[-1].split(".")[2]}.png')
    plt.close()

plt.ion()

/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_000000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_020000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_040000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_060000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_080000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_100000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_120000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_140000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_160000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_180000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_200000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_220000_TAI.3.magnetogram.fits
/hom